Week 14 · Day 4 — Parameter-Efficient Methods (LoRA, Adapters)
Why this matters

Full fine-tuning is powerful but expensive. With billions of parameters, you often can’t update the whole model. LoRA (Low-Rank Adaptation) and adapters let you fine-tune small, injected modules while freezing most of the model — keeping training cheap but effective.

Theory Essentials

PEFT (Parameter-Efficient Fine-Tuning): Freeze base model, train small add-ons.

Adapters: Extra layers in between; only adapters train.

LoRA: Replace weight matrices 
𝑊
W with 
𝑊
+
𝐵
𝐴
W+BA where 
𝐵
,
𝐴
B,A are low-rank → far fewer trainable params.

Benefits:

Much lower compute/memory.

Reusable base model across tasks.

Multiple task adapters can be swapped in/out.

In [6]:
# Setup
from datasets import load_dataset
from transformers import AutoModelForSequenceClassification, AutoTokenizer, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model, TaskType
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

# Dataset
dataset = load_dataset("imdb")
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

def tokenize(batch):
    return tokenizer(batch["text"], padding="max_length", truncation=True, max_length=32)

tokenized = dataset.map(tokenize, batched=True).remove_columns(["text"])
tokenized = tokenized.rename_column("label", "labels")
tokenized.set_format("torch")

small_train = tokenized["train"].shuffle(seed=42).select(range(5000))
small_test = tokenized["test"].shuffle(seed=42).select(range(500))

# Base model
model = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=2)

# LoRA config
lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=16,                # rank
    lora_alpha=32,
    lora_dropout=0.1,
    target_modules=["q_lin", "v_lin"]
)

# Wrap model with LoRA
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# Metrics
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    acc = accuracy_score(labels, preds)
    prec, rec, f1, _ = precision_recall_fscore_support(labels, preds, average="binary")
    return {"accuracy": acc, "precision": prec, "recall": rec, "f1": f1}

# Training
args = TrainingArguments(
    output_dir="./lora-results",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=1,
    logging_dir="./logs",
    logging_steps=50,
    save_steps=10_000,      # big number ≈ "no saving"
    save_total_limit=1
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=small_train,
    eval_dataset=small_test,
    compute_metrics=compute_metrics,
)

trainer.train()
print(trainer.evaluate())


Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


trainable params: 887,042 || all params: 67,842,052 || trainable%: 1.3075


c:\AI-Mastery\venv\Lib\site-packages\torch\utils\data\dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss
50,0.685600
100,0.680200
150,0.664700
200,0.628800
250,0.603600
300,0.599200
350,0.579500
400,0.591600
450,0.554100
500,0.588100


c:\AI-Mastery\venv\Lib\site-packages\torch\utils\data\dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


{'eval_loss': 0.5778067708015442, 'eval_accuracy': 0.698, 'eval_precision': 0.6643598615916955, 'eval_recall': 0.7804878048780488, 'eval_f1': 0.7177570093457943, 'eval_runtime': 14.4954, 'eval_samples_per_second': 34.494, 'eval_steps_per_second': 4.346, 'epoch': 1.0}


1) Core (10–15 min)
Task: Compare number of trainable parameters in LoRA vs full fine-tuning.

trainable params: 739,586 || all params: 67,694,596 || trainable%: 1.0925

2) Practice (10–15 min)
Task: Change r=8 → r=16. Check if accuracy improves.

r=8: {'eval_loss': 0.5768918395042419, 'eval_accuracy': 0.7, 'eval_precision': 0.6690140845070423, 'eval_recall': 0.7723577235772358, 'eval_f1': 0.7169811320754716, 'eval_runtime': 16.0862, 'eval_samples_per_second': 31.083, 'eval_steps_per_second': 3.916, 'epoch': 1.0}

r=16: {'eval_loss': 0.5778067708015442, 'eval_accuracy': 0.698, 'eval_precision': 0.6643598615916955, 'eval_recall': 0.7804878048780488, 'eval_f1': 0.7177570093457943, 'eval_runtime': 14.4954, 'eval_samples_per_second': 34.494, 'eval_steps_per_second': 4.346, 'epoch': 1.0}

3) Stretch (optional, 10–15 min)
Task: Save your LoRA adapter separately, then reload it.

In [7]:
model.save_pretrained("lora-imdb")
from peft import PeftModel
base = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=2)
lora_model = PeftModel.from_pretrained(base, "lora-imdb")


Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Mini-Challenge (≤40 min)

Task: Train with LoRA for 2 epochs and compare results against yesterday’s full fine-tuning.
Acceptance Criteria: Report metrics table (accuracy, precision, recall, F1) and highlight whether LoRA matches full fine-tuning performance with far fewer trainable params.

In [8]:
# Setup
from datasets import load_dataset
from transformers import AutoModelForSequenceClassification, AutoTokenizer, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model, TaskType
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

# Dataset
dataset = load_dataset("imdb")
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

def tokenize(batch):
    return tokenizer(batch["text"], padding="max_length", truncation=True, max_length=32)

tokenized = dataset.map(tokenize, batched=True).remove_columns(["text"])
tokenized = tokenized.rename_column("label", "labels")
tokenized.set_format("torch")

small_train = tokenized["train"].shuffle(seed=42).select(range(5000))
small_test = tokenized["test"].shuffle(seed=42).select(range(500))

# Base model
model = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=2)

# LoRA config
lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=8,                # rank
    lora_alpha=32,
    lora_dropout=0.1,
    target_modules=["q_lin", "v_lin"]
)

# Wrap model with LoRA
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# Metrics
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    acc = accuracy_score(labels, preds)
    prec, rec, f1, _ = precision_recall_fscore_support(labels, preds, average="binary")
    return {"accuracy": acc, "precision": prec, "recall": rec, "f1": f1}

# Training
args = TrainingArguments(
    output_dir="./lora-results",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=2,
    logging_dir="./logs",
    logging_steps=50,
    save_steps=10_000,      # big number ≈ "no saving"
    save_total_limit=1
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=small_train,
    eval_dataset=small_test,
    compute_metrics=compute_metrics,
)

trainer.train()
print(trainer.evaluate())


Map:   0%|          | 0/50000 [00:00<?, ? examples/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


trainable params: 739,586 || all params: 67,694,596 || trainable%: 1.0925


c:\AI-Mastery\venv\Lib\site-packages\torch\utils\data\dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss
50,0.695000
100,0.681000
150,0.666000
200,0.635400
250,0.603800
300,0.593400
350,0.575400
400,0.584800
450,0.557600
500,0.581800


c:\AI-Mastery\venv\Lib\site-packages\torch\utils\data\dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


{'eval_loss': 0.5666990876197815, 'eval_accuracy': 0.7, 'eval_precision': 0.6666666666666666, 'eval_recall': 0.7804878048780488, 'eval_f1': 0.7191011235955056, 'eval_runtime': 14.8352, 'eval_samples_per_second': 33.704, 'eval_steps_per_second': 4.247, 'epoch': 2.0}


Notes / Key Takeaways

Full fine-tune: huge cost, max flexibility.

LoRA/adapters: small cost, nearly as effective.

LoRA adds low-rank matrices in attention layers.

You can train multiple adapters for different tasks and reuse the same base.

Hugging Face peft library makes PEFT simple.

Reflection

Why is LoRA more efficient than full fine-tuning?

When would you still choose full fine-tuning over parameter-efficient methods?

Why is LoRA more efficient than full fine-tuning?

LoRA freezes the original model weights and only learns small low-rank adapters.

This means far fewer trainable parameters → lower memory, faster training, and you can store/share adapters cheaply instead of full models.

When would you still choose full fine-tuning over parameter-efficient methods?

When you have lots of data and compute, and want maximum accuracy.

If the new task is very different from pretraining (e.g., medical or legal language far from general domain).

When you need to deploy a single specialized model and don’t care about adapter modularity or storage.